# Co-Plan vs. Baseline — Preregistered Study Analysis Notebook

Implements the preregistered analysis plan (`/eval/PREREGISTRATION.md`):
1. Loads all per-run JSONL logs from `/eval/raw_logs/` and **fails loudly** (`MockLLMDataError`) if any session used `llm_provider == "mock"`.
2. Computes all 10 metrics: quality, completion time, turns/actions, time-to-recover, override rate, rework count, error rate, NASA-TLX, Jian trust, and perceived control/agency.
3. Runs Shapiro-Wilk normality checks -> paired t-test (Cohen's $d_z$) or Wilcoxon signed-rank (rank-biserial $r$) + 95% bootstrap CIs.
4. Applies Holm-Bonferroni correction across H1–H4 (including H4 non-inferiority margin $\Delta = -0.75$).
5. Runs AB/BA order & carryover checks and blind dual-rater Cohen's $\kappa$ (unweighted + quadratic weighted).
6. Plots **full distributions** (paired slopegraphs + boxplots with overlaid participant points — never bare mean bars).

> **Pipeline Self-Test Mode:** Until real human study data is collected, Cell 3 runs on clearly labelled **`SYNTHETIC - pipeline test only`** data to verify every computation.

In [ ]:
import json
from pathlib import Path
from study_analysis import (
    MockLLMDataError,
    extract_behavioral_metrics,
    generate_synthetic_pipeline_test_data,
    load_study_logs,
    run_full_hypothesis_suite,
)

RAW_LOGS_DIR = Path('../raw_logs')
USE_SYNTHETIC_PIPELINE_TEST = True  # Set False for real study data!

if not USE_SYNTHETIC_PIPELINE_TEST:
    # FAILS LOUDLY if ANY log has llm_provider == 'mock'
    raw_events = load_study_logs(RAW_LOGS_DIR, fail_on_mock=True)
    behavioral = extract_behavioral_metrics(raw_events)
    print(f'Loaded {len(raw_events)} real study events across {len(behavioral)} sessions.')
else:
    print('*** RUNNING ON SYNTHETIC - pipeline test only DATA (NOT real participant data) ***')

In [ ]:
# Execute H1-H4 paired tests, Holm-Bonferroni correction, order/carryover checks, and Cohen's kappa
synth = generate_synthetic_pipeline_test_data(n_participants=12, seed=2026)
report = run_full_hypothesis_suite(
    synth['paired_rows'],
    synth['rater_a'],
    synth['rater_b'],
    dataset_label='SYNTHETIC - pipeline test only',
)
print(json.dumps(report, indent=2))

In [ ]:
# Plot full distributions (paired slope lines + boxplots + jittered individual points)
try:
    import matplotlib.pyplot as plt
    import numpy as np

    rows = synth['paired_rows']
    metrics = [
        ('tlx_baseline', 'tlx_coplan', 'H1: Raw NASA-TLX (lower is better)'),
        ('ttr_ms_baseline', 'ttr_ms_coplan', 'H2: Time to Recover ms (lower is better)'),
        ('control_baseline', 'control_coplan', 'H3: Perceived Control 1-7 (higher is better)'),
        ('quality_baseline', 'quality_coplan', 'H4: Blind Quality 0-10 (non-inferiority)'),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    fig.suptitle('SYNTHETIC - pipeline test only (NOT real participant data)', fontsize=13, fontweight='bold', color='darkred')

    for ax, (b_key, c_key, title) in zip(axes.flat, metrics):
        b_vals = [r[b_key] for r in rows]
        c_vals = [r[c_key] for r in rows]
        ax.boxplot([b_vals, c_vals], positions=[0, 1], widths=0.35, showfliers=False)
        for bv, cv in zip(b_vals, c_vals):
            ax.plot([0, 1], [bv, cv], color='#64748b', alpha=0.45, linewidth=1.2)
        ax.scatter(np.zeros(len(b_vals)), b_vals, color='#334155', zorder=3, label='Baseline')
        ax.scatter(np.ones(len(c_vals)), c_vals, color='#4f46e5', zorder=3, label='Co-Plan')
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Baseline', 'Co-Plan'])
        ax.set_title(title, fontsize=10)
        ax.grid(axis='y', linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.show()
except ImportError:
    print('Install matplotlib and numpy to render full-distribution paired plots.')